In [1]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig
)

from src.model import XMistralForCausalLM

model_name = "Hannibal046/xrag-7b"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding_side = 'left',
    add_eos_token=False, ## import to include this!
    use_fast=False,
)

config = AutoConfig.from_pretrained(model_name)

 ## load llm
config = AutoConfig.from_pretrained(model_name)
MODEL_CLASS = eval(config.architectures[0])
model = MODEL_CLASS.from_pretrained(
    model_name,
    torch_dtype = torch.bfloat16,
    device_map='auto',
    offload_folder="./offload",
    low_cpu_mem_usage = True,
)

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [19]:
p = model.projector
dummy_input = torch.randn(
    10, model.config.retriever_hidden_size,
    device=model.device,
    dtype=model.dtype
)

dummy_input.shape

torch.Size([10, 4096])

In [20]:
with torch.no_grad():
    _ = p(dummy_input)

In [21]:
projector_weights = p.state_dict()

In [23]:
projector_weights

OrderedDict([('projector.0.weight',
              tensor(..., device='meta', size=(4096, 4096), dtype=torch.bfloat16)),
             ('projector.0.bias',
              tensor(..., device='meta', size=(4096,), dtype=torch.bfloat16)),
             ('projector.2.weight',
              tensor(..., device='meta', size=(4096, 4096), dtype=torch.bfloat16)),
             ('projector.2.bias',
              tensor(..., device='meta', size=(4096,), dtype=torch.bfloat16))])

In [11]:

torch.save(projector_weights, "projectorweights/projector_weights.pth")

In [4]:
model.projector.projector

Sequential(
  (0): Linear(in_features=4096, out_features=4096, bias=True)
  (1): GELU(approximate='none')
  (2): Linear(in_features=4096, out_features=4096, bias=True)
)

In [5]:
import torch.nn as nn
import re
from types import SimpleNamespace
import torch

In [6]:
standalone_projector_config = SimpleNamespace(
    projector_type=model.config.projector_type,
    retriever_hidden_size=model.config.retriever_hidden_size,
    hidden_size=model.config.hidden_size
)

In [9]:
model.config.hidden_size

4096

In [39]:
from src.distill.models.projector_xmistral_modeling import Projector

untrained_projector = Projector(standalone_projector_config).to(model.device).to(model.dtype)
untrained_projector.eval()

Projector(
  (projector): Sequential(
    (0): Linear(in_features=4096, out_features=4096, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=4096, out_features=4096, bias=True)
  )
)

In [40]:
dummy_input = torch.randn(
    10, standalone_projector_config.retriever_hidden_size,
    device=model.device,
    dtype=model.dtype
)

dummy_input.shape

torch.Size([10, 4096])

In [41]:
with torch.no_grad():
    output_untrained = untrained_projector(dummy_input)

print(output_untrained.shape)

torch.Size([10, 4096])


In [42]:
trained_projector_orginial = model.projector

In [43]:
trained_projector_orginial.eval()

Projector(
  (projector): Sequential(
    (0): Linear(in_features=4096, out_features=4096, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=4096, out_features=4096, bias=True)
  )
)

In [44]:
with torch.no_grad():
    output_trained_original = trained_projector_orginial(dummy_input)

print(output_trained_original.shape)

torch.Size([10, 4096])


In [45]:
trained_projector = untrained_projector
trained_projector.load_state_dict(trained_projector_orginial.state_dict())

<All keys matched successfully>

In [47]:
with torch.no_grad():
    output_trained = trained_projector(dummy_input)

print(output_trained.shape)

torch.Size([10, 4096])


In [48]:
are_outputs_equal = torch.allclose(output_trained_original, output_trained, atol=1e-5)
are_untrained_different = not torch.allclose(output_untrained, output_trained, atol=1e-5)

print("\n--- EVALUATION ---")
if are_outputs_equal:
    print("✅ SUCCESS: The pre-trained and new standalone projector outputs are identical.")
else:
    print("❌ FAILURE: The outputs are different. Weight transfer failed.")

if are_untrained_different:
    print("✅ SUCCESS: The untrained output is different from the final output, as expected.")
else:
    print("❌ FAILURE: The untrained and trained outputs are the same, which is unexpected.")


--- EVALUATION ---
✅ SUCCESS: The pre-trained and new standalone projector outputs are identical.
✅ SUCCESS: The untrained output is different from the final output, as expected.
